<a href="https://colab.research.google.com/github/stephanyIS/tfm-prediccion-fuga-ml-bancario/blob/main/Modelado_de_datos_Seminario_de_Investigaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline de Modelado - Prediccion de Fuga de Clientes (Customer Churn)
# TFM - Master en Analisis y Visualizacion de Datos Masivos (UNIR)

Orquestador en Python (scikit-learn) del pipeline descrito en la Seccion 3
y ejecutado en la Seccion 4 del documento. Equivalente funcional al flujo
de KNIME documentado en el Anexo C.

Uso: python pipeline_modelado.py
Requiere: Churn_Modelling.csv en el mismo directorio.

In [ ]:
import sqlite3
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import (recall_score, precision_score, f1_score,
                              roc_auc_score, confusion_matrix, accuracy_score)

RNG = 42
np.random.seed(RNG)
INPUT_FILE = "/Churn_Modelling.csv"
DB_FILE = "churn_pipeline.sqlite"

# =========================================================================
# 1. CARGA Y LIMPIEZA (Seccion 3.3.1 / 3.3.2)
# =========================================================================
df = pd.read_csv(INPUT_FILE)
df = df.drop(columns=["RowNumber", "CustomerId", "Surname"])
df = df.dropna()  # el dataset no trae nulos; se deja como medida de robustez


In [ ]:
# 2. CODIFICACION DE CATEGORICAS (Seccion 3.3.5)
#    No depende de estadisticos de muestra -> se puede aplicar antes de partir
df = pd.get_dummies(df, columns=["Geography", "Gender"], drop_first=False)

y = df["Exited"].values
X_df = df.drop(columns=["Exited"])

In [ ]:
# 3. PARTICION ESTRATIFICADA 80/20 (Seccion 3.5.1 / 4.2)
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.20, stratify=y, random_state=RNG)

print(f"Train: {len(y_train)} (churn={y_train.mean():.4f})  "
      f"Test: {len(y_test)} (churn={y_test.mean():.4f})")



Train: 8000 (churn=0.2037)  Test: 2000 (churn=0.2035)


In [ ]:
# 4. WINSORIZACION DE Age (Seccion 3.3.3) -- AJUSTADA SOLO CON TRAIN
#    Equivalente a Numeric Outliers (Learner) + Numeric Outliers (Apply)
age_low, age_high = X_train_df["Age"].quantile([0.01, 0.99])
X_train_df["Age"] = X_train_df["Age"].clip(age_low, age_high)
X_test_df["Age"] = X_test_df["Age"].clip(age_low, age_high)  # limites del TRAIN
print(f"Limites de winsorizacion (train): Age in [{age_low:.1f}, {age_high:.1f}]")




Limites de winsorizacion (train): Age in [21.0, 72.0]


In [ ]:
# 5. FEATURE ENGINEERING (Seccion 3.3.4) -- formulas fijas, sin leakage
def feature_engineer(d):
    d = d.copy()
    d["Balance_Salary_Ratio"] = d["Balance"] / d["EstimatedSalary"].replace(0, 1)
    d["Products_Balance_Interaction"] = d["NumOfProducts"] * d["Balance"]
    d["Has_Balance"] = (d["Balance"] > 0).astype(int)
    bins = [17, 30, 45, 60, 100]
    labels = ["18-30", "31-45", "46-60", "61+"]
    d["Age_Group"] = pd.cut(d["Age"], bins=bins, labels=labels)
    return d

X_train_df = feature_engineer(X_train_df)
X_test_df = feature_engineer(X_test_df)

X_train_df = pd.get_dummies(X_train_df, columns=["Age_Group"], drop_first=False)
X_test_df = pd.get_dummies(X_test_df, columns=["Age_Group"], drop_first=False)
X_test_df = X_test_df.reindex(columns=X_train_df.columns, fill_value=0)  # alinear columnas

feature_names = X_train_df.columns.tolist()
X_train = X_train_df.values.astype(float)
X_test = X_test_df.values.astype(float)



In [ ]:
# 6. NORMALIZACION Z-SCORE (Seccion 3.3.6) -- fit SOLO en train
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [ ]:
# 7. SMOTE (Seccion 3.3.7) -- confinado EXCLUSIVAMENTE al train
def smote(X, y, k=5, random_state=42):
    rng = np.random.RandomState(random_state)
    X_min = X[y == 1]
    n_needed = (y == 0).sum() - (y == 1).sum()
    if n_needed <= 0 or len(X_min) <= 1:
        return X, y
    k = min(k, len(X_min) - 1)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X_min)
    _, idx = nn.kneighbors(X_min)
    synth = []
    for _ in range(n_needed):
        i = rng.randint(0, len(X_min))
        neighbor = idx[i][rng.randint(1, k + 1)]
        gap = rng.rand()
        synth.append(X_min[i] + gap * (X_min[neighbor] - X_min[i]))
    X_syn = np.vstack(synth)
    y_syn = np.ones(len(X_syn))
    return np.vstack([X, X_syn]), np.concatenate([y, y_syn])

X_train_bal, y_train_bal = smote(X_train_s, y_train, k=5, random_state=RNG)
print(f"Train tras SMOTE: {len(y_train_bal)} (churn={y_train_bal.mean():.4f})")



Train tras SMOTE: 12740 (churn=0.5000)


In [ ]:
# 8. VALIDACION CRUZADA ESTRATIFICADA k=5 (Seccion 3.2.3/3.2.5 / 4.4)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RNG)

def cv_score(model, X, y):
    aucs, recalls = [], []
    for tr_idx, val_idx in skf.split(X, y):
        model.fit(X[tr_idx], y[tr_idx])
        proba = model.predict_proba(X[val_idx])[:, 1]
        aucs.append(roc_auc_score(y[val_idx], proba))
        recalls.append(recall_score(y[val_idx], model.predict(X[val_idx])))
    return np.mean(aucs), np.std(aucs), np.mean(recalls), np.std(recalls)

lr = LogisticRegression(max_iter=1000, random_state=RNG)
rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=3,
                             random_state=RNG)

lr_auc, lr_auc_sd, lr_rec, lr_rec_sd = cv_score(lr, X_train_bal, y_train_bal)
rf_auc, rf_auc_sd, rf_rec, rf_rec_sd = cv_score(rf, X_train_bal, y_train_bal)
print(f"\nCV (k=5) LR: AUC={lr_auc:.3f}+-{lr_auc_sd:.3f}  Recall={lr_rec:.3f}+-{lr_rec_sd:.3f}")
print(f"CV (k=5) RF: AUC={rf_auc:.3f}+-{rf_auc_sd:.3f}  Recall={rf_rec:.3f}+-{rf_rec_sd:.3f}")


CV (k=5) LR: AUC=0.800+-0.001  Recall=0.706+-0.009
CV (k=5) RF: AUC=0.899+-0.003  Recall=0.789+-0.017


In [ ]:
# 9. ENTRENAMIENTO FINAL Y EVALUACION EN TEST (holdout real, 20%)
lr.fit(X_train_bal, y_train_bal)
rf.fit(X_train_bal, y_train_bal)

results = {}
for name, model in [("Regresion Logistica", lr), ("Random Forest", rf)]:
    proba = model.predict_proba(X_test_s)[:, 1]
    pred = model.predict(X_test_s)
    results[name] = {
        "AUC-ROC": roc_auc_score(y_test, proba),
        "Recall": recall_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "Accuracy": accuracy_score(y_test, pred),
        "CM": confusion_matrix(y_test, pred),
    }

print(f"\n=== EVALUACION EN TEST (holdout, 20% -> {len(y_test)} registros) ===")
for name, r in results.items():
    print(f"\n{name}:")
    for k, v in r.items():
        if k != "CM":
            print(f"  {k}: {v:.4f}")
    print(f"  Matriz de confusion [[VN,FP],[FN,VP]]:\n{r['CM']}")



=== EVALUACION EN TEST (holdout, 20% -> 2000 registros) ===

Regresion Logistica:
  AUC-ROC: 0.8015
  Recall: 0.7199
  Precision: 0.4198
  F1: 0.5303
  Accuracy: 0.7405
  Matriz de confusion [[VN,FP],[FN,VP]]:
[[1188  405]
 [ 114  293]]

Random Forest:
  AUC-ROC: 0.8590
  Recall: 0.7027
  Precision: 0.5532
  F1: 0.6190
  Accuracy: 0.8240
  Matriz de confusion [[VN,FP],[FN,VP]]:
[[1362  231]
 [ 121  286]]


In [ ]:
# 10. INTERPRETACION: coeficientes LR / importancia RF
coefs = pd.Series(lr.coef_[0], index=feature_names).sort_values(key=abs, ascending=False)
print("\nTop coeficientes Regresion Logistica (estandarizados):")
print(coefs.head(10).round(3))

importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
print("\nTop importancia de variables Random Forest (Gini):")
print(importances.head(10).round(3))



Top coeficientes Regresion Logistica (estandarizados):
Age                             0.989
Products_Balance_Interaction    0.768
Balance                        -0.528
IsActiveMember                 -0.489
NumOfProducts                  -0.481
Age_Group_61+                  -0.436
Geography_Germany               0.228
Age_Group_46-60                 0.186
Gender_Female                   0.141
Gender_Male                    -0.141
dtype: float64

Top importancia de variables Random Forest (Gini):
NumOfProducts                   0.239
Age                             0.232
Age_Group_46-60                 0.149
IsActiveMember                  0.102
Products_Balance_Interaction    0.043
Geography_Germany               0.040
Age_Group_18-30                 0.039
Balance                         0.034
Age_Group_31-45                 0.025
Balance_Salary_Ratio            0.021
dtype: float64


In [ ]:
# 11. SENSIBILIDAD: barrido de umbral (Random Forest)
print("\n=== Sensibilidad de Random Forest al umbral de decision ===")
proba_rf = rf.predict_proba(X_test_s)[:, 1]
for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred_thr = (proba_rf >= thr).astype(int)
    print(f"  umbral={thr:.1f}  Recall={recall_score(y_test, pred_thr):.4f}  "
          f"Precision={precision_score(y_test, pred_thr, zero_division=0):.4f}")



=== Sensibilidad de Random Forest al umbral de decision ===
  umbral=0.3  Recall=0.9287  Precision=0.3198
  umbral=0.4  Recall=0.8157  Precision=0.4034
  umbral=0.5  Recall=0.7027  Precision=0.5532
  umbral=0.6  Recall=0.5872  Precision=0.6425
  umbral=0.7  Recall=0.4300  Precision=0.7543


In [ ]:

# 12. SENSIBILIDAD: ablation SMOTE
rf_noSmote = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=3,
                                     random_state=RNG)
rf_noSmote.fit(X_train_s, y_train)
proba_ns = rf_noSmote.predict_proba(X_test_s)[:, 1]
pred_ns = rf_noSmote.predict(X_test_s)
print(f"\nRF SIN SMOTE -> AUC={roc_auc_score(y_test, proba_ns):.4f}  "
      f"Recall={recall_score(y_test, pred_ns):.4f}  "
      f"Precision={precision_score(y_test, pred_ns, zero_division=0):.4f}")
print(f"RF CON SMOTE -> AUC={results['Random Forest']['AUC-ROC']:.4f}  "
      f"Recall={results['Random Forest']['Recall']:.4f}  "
      f"Precision={results['Random Forest']['Precision']:.4f}")



RF SIN SMOTE -> AUC=0.8591  Recall=0.3956  Precision=0.7970
RF CON SMOTE -> AUC=0.8590  Recall=0.7027  Precision=0.5532


In [ ]:
# 13. PERSISTENCIA EN SQLITE (Seccion 2.5.2)
conn = sqlite3.connect(DB_FILE)
train_out = X_train_df.copy(); train_out["Exited"] = y_train
test_out = X_test_df.copy(); test_out["Exited"] = y_test
test_out["proba_rf"] = proba_rf
test_out["pred_rf"] = rf.predict(X_test_s)
train_out.to_sql("train_preprocessed", conn, if_exists="replace", index=False)
test_out.to_sql("test_predictions", conn, if_exists="replace", index=False)
conn.close()
print(f"\nResultados persistidos en {DB_FILE} (tablas: train_preprocessed, test_predictions)")


Resultados persistidos en churn_pipeline.sqlite (tablas: train_preprocessed, test_predictions)
